In [0]:
# =============================================================================
# PIPELINE ETL - FÁBRICA 2025
# Arquitetura Medallion: Bronze → Silver → Gold
# Desenvolvido para Databricks | Linguagem: PySpark + Python
# Baseado nas boas práticas do livro "Understanding ETL"
# =============================================================================
#
# ESTRUTURA DO PIPELINE:
#   1. BRONZE  → Ingestão bruta dos dados (sem transformação)
#   2. SILVER  → Limpeza, tipagem e enriquecimento
#   3. GOLD    → Tabelas analíticas para gestores
#       - gold_qualidade_turno   : Indicadores de qualidade por turno/equipe
#       - gold_desperdicio_producao : Análise de desperdício e perdas
#
# FONTES DE DADOS:
#   - quality_daily.jsonl      → Produção e qualidade por turno/máquina
#   - downtime_events.jsonl    → Eventos de parada de máquina
#   - safety_daily.json        → FPY (First Pass Yield) por turno
#   - production_shift.csv     → Metas mensais de produção
#   - hr_employees.csv         → Escala de turnos e cadastro de funcionários
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType, TimestampType
)

# Iniciando a sessão Spark (no Databricks já existe automaticamente como 'spark')
# Se rodar localmente, descomente a linha abaixo:
# spark = SparkSession.builder.appName("Pipeline_Fabrica").getOrCreate()

# Caminho base onde os arquivos estão armazenados no Databricks (DBFS ou Volume)
# Ajuste conforme seu ambiente:
BASE_PATH = "workspace.qap_2025_raw.volume/"

# Caminho onde as tabelas Delta serão salvas
DELTA_PATH = "workspace.qap_2025_raw.volume"


# =============================================================================
# ██████╗ ██████╗  ██████╗ ███╗   ██╗███████╗███████╗
# ██╔══██╗██╔══██╗██╔═══██╗████╗  ██║╚══███╔╝██╔════╝
# ██████╔╝██████╔╝██║   ██║██╔██╗ ██║  ███╔╝ █████╗
# ██╔══██╗██╔══██╗██║   ██║██║╚██╗██║ ███╔╝  ██╔══╝
# ██████╔╝██║  ██║╚██████╔╝██║ ╚████║███████╗███████╗
# ╚═════╝ ╚═╝  ╚═╝ ╚═════╝ ╚═╝  ╚═══╝╚══════╝╚══════╝
#
# CAMADA BRONZE — Ingestão Bruta
# =============================================================================
#
# 📖 POR QUE BRONZE?
# O livro "Understanding ETL" recomenda preservar os dados originais sem
# nenhuma transformação. Isso garante:
#   ✔ Rastreabilidade: sempre podemos voltar à fonte original
#   ✔ Reprocessamento: se a lógica mudar, reprocessamos a partir do bronze
#   ✔ Auditoria: dados brutos servem como evidência
#
# 📖 POR QUE PYSPARK E NÃO PANDAS?
# Para dados de fábrica que crescem diariamente, o PySpark escala melhor.
# O Pandas carrega tudo na memória de uma máquina. O Spark distribui o
# processamento em cluster — essencial em ambiente Databricks.
#
# 📖 POR QUE DELTA LAKE?
# O Delta Lake (formato padrão do Databricks) adiciona:
#   ✔ ACID transactions (não corrompe dados em caso de falha)
#   ✔ Time travel (ver versões anteriores da tabela)
#   ✔ Schema enforcement (rejeita dados com estrutura errada)
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# BRONZE 1: quality_daily (dados de qualidade por turno)
# Fonte: CSV (apesar da extensão .jsonl, o arquivo é CSV)
# ─────────────────────────────────────────────────────────────────────────────

print(">>> [BRONZE] Lendo quality_daily...")

# Lemos com inferSchema=True para deixar o Spark detectar os tipos automaticamente
# No Bronze, aceitamos os dados como vieram — sem forçar tipos ainda
df_bronze_quality = (
    spark.read
    .option("header", "true")       # primeira linha é o cabeçalho
    .option("inferSchema", "true")  # Spark tenta detectar os tipos
    .csv(f"{BASE_PATH}/quality_daily.jsonl")
)

# Adicionamos metadados de controle — boa prática do livro (cap. Monitoring)
# Isso permite saber quando o dado foi ingerido, facilitando auditoria
df_bronze_quality = df_bronze_quality.withColumn(
    "ingestion_ts", F.current_timestamp()   # timestamp de quando foi carregado
).withColumn(
    "source_file", F.lit("quality_daily.jsonl")  # nome do arquivo de origem
)

# Salvamos como tabela Delta no Bronze
# mode("overwrite") → substitui tudo a cada execução (carga full)
# Para produção, considere mode("append") com controle de duplicatas
df_bronze_quality.write.format("delta").mode("overwrite").saveAsTable(    f"{DELTA_PATH}.bronze_quality_daily"
)

print(f"    Registros ingeridos: {df_bronze_quality.count()}")
print("    Colunas:", df_bronze_quality.columns)


# ─────────────────────────────────────────────────────────────────────────────
# BRONZE 2: downtime_events (eventos de parada de máquina)
# Fonte: CSV (apesar da extensão .jsonl)
# ─────────────────────────────────────────────────────────────────────────────

print("\n>>> [BRONZE] Lendo downtime_events...")

df_bronze_downtime = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_PATH}/downtime_events.jsonl")
)

df_bronze_downtime = df_bronze_downtime.withColumn(
    "ingestion_ts", F.current_timestamp()
).withColumn(
    "source_file", F.lit("downtime_events.jsonl")
)

df_bronze_downtime.write.format("delta").mode("overwrite").saveAsTable(
    f"{DELTA_PATH}.bronze_downtime_events"
)

print(f"    Registros ingeridos: {df_bronze_downtime.count()}")


# ─────────────────────────────────────────────────────────────────────────────
# BRONZE 3: safety_daily (FPY - First Pass Yield por turno)
# Fonte: JSON Lines (um objeto JSON por linha)
# ─────────────────────────────────────────────────────────────────────────────

print("\n>>> [BRONZE] Lendo safety_daily...")

# 📖 POR QUE multiLine=False?
# O arquivo tem um JSON por linha (JSON Lines / JSONL).
# multiLine=True seria para um único objeto JSON gigante em várias linhas.
# Usar o modo errado causaria erro de parse — importante entender a diferença!
df_bronze_safety = (
    spark.read
    .option("multiLine", "false")
    .json(f"{BASE_PATH}/safety_daily.json")
)

df_bronze_safety = df_bronze_safety.withColumn(
    "ingestion_ts", F.current_timestamp()
).withColumn(
    "source_file", F.lit("safety_daily.json")
)

df_bronze_safety.write.format("delta").mode("overwrite").saveAsTable(
    f"{DELTA_PATH}.bronze_safety_daily"
)

print(f"    Registros ingeridos: {df_bronze_safety.count()}")


# ─────────────────────────────────────────────────────────────────────────────
# BRONZE 4: production_shift (metas mensais de produção)
# ─────────────────────────────────────────────────────────────────────────────

print("\n>>> [BRONZE] Lendo production_shift...")

df_bronze_shift = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{BASE_PATH}/production_shift.csv")
)

df_bronze_shift = df_bronze_shift.withColumn(
    "ingestion_ts", F.current_timestamp()
).withColumn(
    "source_file", F.lit("production_shift.csv")
)

df_bronze_shift.write.format("delta").mode("overwrite").saveAsTable(
    f"{DELTA_PATH}.bronze_production_shift"
)

print(f"    Registros ingeridos: {df_bronze_shift.count()}")


# ─────────────────────────────────────────────────────────────────────────────
# BRONZE 5: hr_employees (escala de turnos + cadastro de funcionários)
# Fonte: CSV com duas tabelas concatenadas (schedule + employees)
# ─────────────────────────────────────────────────────────────────────────────

print("\n>>> [BRONZE] Lendo hr_employees...")

# 📖 ATENÇÃO: Este arquivo tem duas tabelas em sequência no mesmo CSV.
# No Bronze, ingerimos tudo como está — sem separar ainda.
# A separação acontece no Silver (onde fazemos transformações).
# Isso respeita o princípio do Bronze: "dados como vieram da fonte".
df_bronze_hr = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "PERMISSIVE")  # não falha em linhas com problema
    .excel(f"{BASE_PATH}/hr_employees.xlsx")
)

df_bronze_hr = df_bronze_hr.withColumn(
    "ingestion_ts", F.current_timestamp()
).withColumn(
    "source_file", F.lit("hr_employees.csv")
)

df_bronze_hr.write.format("delta").mode("overwrite").saveAsTable(
    f"{DELTA_PATH}.bronze_hr_employees"
)

print(f"    Registros ingeridos: {df_bronze_hr.count()}")

print("\n✅ BRONZE concluído! Todas as fontes foram ingeridas.")


In [0]:
%sql
CREATE SCHEMA workspace.quality_auto_factory_2025.producao_bronze;
CREATE SCHEMA workspace.quality_auto_factory_2025.producao_silver;
CREATE SCHEMA workspace.quality_auto_factory_2025.producao_gold;